# 로컬 LLM

- API 호출 없이 자신의 컴퓨터에서 LLM을 실행해본다
- LangChain의 추상화 덕분에 로컬 모델도 API 모델과 동일한 코드로 사용할 수 있다

## Ollama

로컬에서 LLM을 쉽게 실행할 수 있게 해주는 도구이다. `ollama pull`로 모델을 다운로드하고 `ollama run`으로 실행하면 된다.

- 공식 사이트: https://ollama.com
- Llama(Meta), Qwen(Alibaba), Gemma(Google) 등 다양한 오픈소스 모델을 지원한다
- 로컬 API 서버를 띄워주기 때문에 LangChain 등 외부 도구에서 쉽게 연동할 수 있다

## Hugging Face

오픈소스 AI 모델의 **GitHub** 같은 플랫폼이다. LLM뿐만 아니라 이미지, 음성 등 다양한 AI 모델이 공유되어 있다.

- 공식 사이트: https://huggingface.co
- **Models**: 수십만 개의 사전학습 모델을 검색하고 다운로드할 수 있다
- **Datasets**: 학습용 데이터셋 공유 및 다운로드
- **Spaces**: 모델을 웹에서 바로 체험할 수 있는 데모 공간

Ollama 라이브러리에 없는 모델도 Hugging Face에서 GGUF 형식으로 다운로드하면 Ollama로 실행할 수 있다. 이처럼 모델을 모아놓은 플랫폼을 **Model Hub**라고 부른다.

## Transformers vs Ollama

Hugging Face에서는 `transformers`라는 Python 라이브러리도 제공한다. 이 라이브러리를 사용하면 모델을 Python 코드 안에서 직접 로드하고 추론할 수 있다.

하지만 원본 모델은 용량이 크고 GPU가 사실상 필수이기 때문에, 일반 PC에서 돌리려면 **양자화**(quantization)라는 경량화 과정이 필요하다. 양자화란 모델의 가중치를 더 작은 숫자로 압축하는 것으로, 품질을 약간 희생하는 대신 크기와 속도를 크게 개선한다.

그리고 이 양자화된 모델을 쉽게 돌려주는 도구가 바로 **Ollama**이다. 정리하면:

- Hugging Face에서 모델 다운로드 → 양자화(경량화) → **Ollama로 실행**

즉 Ollama는 경량화된 모델을 편리하게 실행하는 런타임이다. 이 강의에서는 복잡한 환경 설정 없이 바로 로컬 LLM을 체험할 수 있는 Ollama를 사용한다.

### API 모델로 체인 만들기

In [1]:
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {role} 전문가야. 모든 답변은 한국어로 해줘."),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({
    "role": "Python",
    "question": "for 문을 설명해줘",
})

print(result)

`for` 문은 Python에서 반복(iteration)을 수행하는 데 사용되는 제어 구조입니다. 주로 리스트, 튜플, 문자열, 딕셔너리와 같은 iterable(반복 가능한) 객체의 요소를 하나씩 순회하면서 작업을 수행할 때 사용됩니다.

### 기본 문법

```python
for 변수 in iterable:
    # 반복할 코드 블록
```

- `변수`: 현재 반복에서 iterable의 요소를 저장할 변수입니다.
- `iterable`: 반복할 수 있는 객체(예: 리스트, 튜플, 문자열 등)입니다.

### 예제

1. **리스트 반복하기**

```python
fruits = ['사과', '바나나', '체리']
for fruit in fruits:
    print(fruit)
```

위 코드는 리스트 `fruits`의 각 요소를 하나씩 출력합니다. 결과는 다음과 같습니다:

```
사과
바나나
체리
```

2. **문자열 반복하기**

```python
word = "안녕하세요"
for letter in word:
    print(letter)
```

이 코드는 문자열 `word`의 각 문자를 하나씩 출력합니다.

3. **range() 함수와 함께 사용하기**

`range()` 함수를 사용하면 특정 범위의 숫자를 생성할 수 있습니다.

```python
for i in range(5):
    print(i)
```

이 코드는 0부터 4까지의 숫자를 출력합니다.

### 중첩된 for 문

`for` 문은 중첩해서 사용할 수도 있습니다.

```python
for i in range(3):
    for j in range(2):
        print(f"i: {i}, j: {j}")
```

이 코드는 다음과 같은 결과를 출력합니다:

```
i: 0, j: 0
i: 0, j: 1
i: 1, j: 0
i: 1, j: 1
i: 2, j: 0
i: 2, j: 1
```

### 요약

`for` 문은 Python에서 반복 작업을 수행하

### Ollama 설치

공식 사이트(https://ollama.com)에서 설치 파일을 다운로드한다.

- **macOS**: `brew install ollama` 또는 공식 사이트에서 다운로드
- **Windows**: 공식 사이트에서 설치 파일 다운로드

설치 후 터미널에서 모델을 다운로드한다:

```bash
# 모델 다운로드 (0.5b는 가장 가벼운 모델로, CPU에서도 무리 없이 실행된다)
ollama pull qwen2.5:0.5b

# 서버가 꺼져 있다면 직접 실행
ollama serve
```

```bash
# LangChain Ollama 연동 패키지 설치
uv add langchain-ollama
```

In [2]:
from langchain_ollama import ChatOllama

# 모델만 교체하면 된다
local_llm = ChatOllama(model="qwen2.5:0.5b", temperature=0)

# 나머지는 동일
chain = prompt | local_llm | parser

result = chain.invoke({
    "role": "Python",
    "question": "for 문을 설명해줘",
})

print(result)

Python의 for 문은 반복문 중 하나입니다. 이 문법은 다음과 같이 작성할 수 있습니다:

```python
for 변수 in 리스트:
    # 코드를 실행합니다.
```

이 문법에서 `리스트`는 어떤 데이터를 담고 있는 리스트가 될 것입니다. `변수`는 `for` 문을 통해 반복되는 값을 저장하는 변수입니다.

반복문은 여러 개의 작업을 수행하고, 이 작업들은 각각 하나의 `for` 문에 포함됩니다. 예를 들어, 다음과 같이 사용할 수 있습니다:

```python
for i in range(10):
    print(i)
```

이 코드는 0부터 9까지의 숫자를 반복하여 출력합니다.

반복문은 여러 가지 종류가 있습니다. 예를 들어, `while` 문과 `for` 문 사이에는 `if` 문을 사용할 수 있습니다:

```python
i = 1
while i < 5:
    print(i)
    i += 1
```

이 코드는 1부터 4까지의 숫자를 반복하여 출력합니다. 이는 `i`가 1보다 작아질 때까지 계속 반복하고, `i`가 5보다 크면 `break`문을 사용하여 종료됩니다.

반복문은 여러 가지 특징이 있습니다. 예를 들어, `for` 문은 변수에 대한 값을 저장할 수 있으며, `while` 문과 `for` 문 사이에는 `if` 문을 사용할 수 있습니다. 이는 반복되는 작업을 중지하고, 만약 작업이 완료되면 종료됩니다.


### 복잡한 질문으로 비교하기

간단한 질문에서는 차이가 크지 않지만, 추론이 필요한 질문에서는 모델 성능 차이가 뚜렷하게 드러난다.

In [ ]:
complex_question = {
    "role": "Python",
    "question": "재귀를 사용하지 않고 피보나치 수열의 n번째 값을 구하는 함수를 작성하고, 시간 복잡도를 분석해줘",
}

print("=== API 모델 (gpt-4o-mini) ===\n")
api_chain = prompt | llm | parser
print(api_chain.invoke(complex_question))

print("\n\n=== 로컬 모델 (qwen2.5:0.5b) ===\n")
local_chain = prompt | local_llm | parser
print(local_chain.invoke(complex_question))

예시 응답
~~~
=== API 모델 (gpt-4o-mini) ===

재귀를 사용하지 않고 피보나치 수열의 n번째 값을 구하는 함수를 작성할 수 있습니다. 일반적으로 피보나치 수열은 다음과 같이 정의됩니다:

- F(0) = 0
- F(1) = 1
- F(n) = F(n-1) + F(n-2) (n >= 2)

재귀를 사용하지 않고 반복문을 이용하여 피보나치 수열의 n번째 값을 구하는 코드는 다음과 같습니다:

```python
def fibonacci(n):
    if n < 0:
        raise ValueError("n은 0 이상의 정수여야 합니다.")
    elif n == 0:
        return 0
    elif n == 1:
        return 1

    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

# 예시 사용
n = 10
print(f"{n}번째 피보나치 수열의 값: {fibonacci(n)}")
```

### 시간 복잡도 분석
이 함수의 시간 복잡도는 O(n)입니다. 이는 n번째 피보나치 수를 계산하기 위해 반복문이 n-1번 실행되기 때문입니다. 각 반복에서 상수 시간 O(1) 내에 두 개의 변수를 업데이트하므로 전체 시간 복잡도는 O(n)입니다.

### 공간 복잡도 분석
이 함수의 공간 복잡도는 O(1)입니다. 추가적인 메모리 사용은 상수 개수의 변수(a, b)만 필요하므로, 입력 n의 크기에 관계없이 공간 사용량은 일정합니다.

=== 로컬 모델 (qwen2.5:0.5b) ===

Python에서 피보나치 수열을 구하는 함수는 다음과 같이 작성할 수 있습니다:

```python
def fibonacci(n):
    if n <= 1:
        return n
    else:
        return (fibonacci(n-1) + fibonacci(n-2))

# 피보나치 수열의 n번째 값을 구하는 함수
def nth_fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return nth_fibonacci(n - 1) + nth_fibonacci(n - 2)

# 시간 복잡도 분석
import time

start_time = time.time()
fibonacci(5)
end_time = time.time()

print("피보나치 수열의 5번째 값은", fibonacci(5), "입니다.")
print("피보나치 수열의 5번째 값을 구하는 시간: ", end_time - start_time, "초")

start_time = time.time()
nth_fibonacci(5)
end_time = time.time()

print("피보나치 수열의 5번째 값은", nth_fibonacci(5), "입니다.")
print("피보나치 수열의 5번째 값을 구하는 시간: ", end_time - start_time, "초")
```

이 코드는 피보나치 수열의 n번째 값을 구하는 함수 `fibonacci`와 `nth_fibonacci`를 작성했습니다. `fibonacci(n)` 함수는 피보나치 수열의 n번째 값을 반환합니다. `nth_fibonacci(n)` 함수는 n번째 피보나치 수열의 값을 구하는 함수입니다.

피보나치 수열은 다음과 같이 생성할 수 있습니다:
- 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, ...
- 이 수열에서 n번째 피보나치 수는 n+1번째 피보나치 수와 같을 때까지의 값을 구할 수 있습니다.

이 코드는 `time` 모듈을 사용하여 시간 복잡도를 분석합니다. `fibonacci(5)` 함수는 5번째 피보나치 수를 반환하고, `nth_fibonacci(5)` 함수는 5번째 피보나치 수를 구하는 데 사용됩니다.

이 코드의 실행 결과:
- 피보나치 수열의 5번째 값은 8입니다.
- 피보나치 수열의 5번째 값을 구하는 시간: 0.001초

따라서, 이 함수는 피보나치 수열의 n번째 값을 구하는 데 매우 효율적입니다.
~~~


로컬 모델과 API 모델의 차이를 직접 비교해보자.

| 비교 항목 | 로컬 LLM (qwen2.5:0.5b) | API (gpt-4o-mini) |
|-----------|--------------------------|---------------------|
| 속도 | 느림 (CPU 추론) | 빠름 |
| 품질 | 낮음 (파라미터 수 적음) | 높음 |
| 비용 | 무료 | 토큰당 과금 |
| 인터넷 | 불필요 | 필요 |

코드는 모델 객체만 교체하면 되므로 동일하다. 작은 모델로 개발/테스트하고, 실제 서비스에서는 API 모델을 사용하는 전략도 가능하다.